# 01 · Etiquetado de regímenes

Ajusta el HMM gaussiano **solo con el tramo de entrenamiento**, canoniza los estados por volatilidad creciente y deja la etiqueta de régimen que consumen las ventanas.

**Responsable:** Oscar

**Entradas**

- `data/processed/canales.parquet`

**Salidas**

- `data/processed/regimenes.parquet`
- `data/processed/etiquetador_regimenes.pkl`
- `results/figures/regimenes_indice.png`

**Tiempo estimado:** ~3 min en CPU (cinco ajustes de Baum-Welch sobre el tramo de train).

In [ ]:
import sys; sys.path.insert(0, "..")   # permite ejecutar desde notebooks/
import src                              # fija el backend de Keras a PyTorch
from src import config, viz
config.fijar_semillas()
viz.aplicar_estilo()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import datos, regimenes

## Por qué un HMM y por qué solo con train

No existe una etiqueta verdadera de régimen: es una variable latente. El HMM
gaussiano la estima imponiendo persistencia temporal, que es la propiedad que
distingue un régimen de una racha de días volátiles sueltos.

El ajuste usa **exclusivamente** el tramo de entrenamiento. Ajustarlo sobre todo
el histórico filtraría al modelo downstream información del periodo de test a
través de las etiquetas: el etiquetador habría visto 2022 y 2023 al decidir dónde
poner las fronteras entre estados.

In [ ]:
canales = pd.read_parquet(src.DIR_PROCESADO / "canales.parquet")

catalogo = config.cargar_catalogo()
columnas_hmm = catalogo["regimenes"]["features_etiquetado"]
particiones = config.particiones()

features_hmm = canales[columnas_hmm]
train_hmm = features_hmm.loc[:particiones.train_hasta]

print("Indicadores de etiquetado:", columnas_hmm)
print("Tramo de ajuste:", train_hmm.index[0].date(), "→", train_hmm.index[-1].date(),
      "·", len(train_hmm), "sesiones")

## Canonicalización de los estados

Baum-Welch numera los estados en el orden en que los encuentra, que depende de la
inicialización. Sin canonizar, el estado 0 puede ser calma en una ejecución y
crisis en la siguiente, y todo el análisis posterior deja de ser comparable.

El criterio es volatilidad creciente, con desempate por retorno medio **solo**
entre estados de volatilidad parecida. Ordenar por retorno como criterio primario
intercambia estados de forma errática cuando dos regímenes tienen volatilidad
similar.

Se prueban las cinco semillas del catálogo y se conserva el ajuste de mayor
log-verosimilitud, porque Baum-Welch converge a óptimos locales.

In [ ]:
etiquetador = regimenes.EtiquetadorRegimenes.desde_catalogo()
etiquetador.fit(train_hmm)

print(etiquetador.n_estados, "estados · covarianza", etiquetador.covarianza)
print("Permutación crudo → canónico:", etiquetador.orden)
print("Log-verosimilitud en train:", round(etiquetador.modelo.score(train_hmm.to_numpy()), 1))

## Inferencia sobre el histórico completo

El HMM ya está ajustado y congelado; aplicarlo al periodo posterior es inferencia,
no ajuste, y no filtra información.

In [ ]:
regimen_diario = etiquetador.predict(features_hmm)
proba = etiquetador.predict_proba(features_hmm)

print("Reparto diario (todo el histórico):")
print(regimenes.distribucion(regimen_diario, etiquetador.n_estados))
proba.tail(3)

## Caracterización económica de cada estado

Comprobación de que la numeración significa lo que dice: la volatilidad y el
drawdown medios deben empeorar monótonamente del estado 0 al 2, y el retorno medio
debe caer.

In [ ]:
from src.evaluacion import nombres_regimenes

perfil = canales.groupby(regimen_diario.rename("regimen"))[columnas_hmm].mean()
perfil.index = [
    "{} · {}".format(k, n)
    for k, n in enumerate(nombres_regimenes(etiquetador.n_estados))
]
perfil.round(3)

## Matriz de transición

Un HMM útil tiene la diagonal muy dominante: los regímenes duran meses, no días.
Una diagonal por debajo de ~0.9 indica que el modelo está capturando ruido diario
y no estructura de mercado. La matriz se reordena con la misma permutación
canónica que las etiquetas.

In [ ]:
orden = etiquetador.orden
transicion = pd.DataFrame(
    etiquetador.modelo.transmat_[np.ix_(orden, orden)],
    index=nombres_regimenes(etiquetador.n_estados),
    columns=nombres_regimenes(etiquetador.n_estados),
).rename_axis(index="desde", columns="hacia")

duracion = pd.Series(
    1.0 / (1.0 - np.diag(transicion.to_numpy())),
    index=transicion.index,
    name="duración media (sesiones)",
)

display(transicion.round(3))
duracion.round(1)

## Agregación al horizonte de predicción

Lo que se predice no es el régimen de hoy sino el **dominante en los 21 días
siguientes**. Esa es la etiqueta que hace la tarea no trivial: el régimen actual
es prácticamente observable a partir de los propios canales de entrada.

El método `modal` toma el régimen más frecuente de la ventana futura. La
alternativa `maximo` etiquetaría como crisis cualquier ventana que la roce, lo que
infla artificialmente la clase minoritaria.

In [ ]:
regimen_futuro = regimenes.regimen_dominante(
    regimen_diario,
    horizonte=config.ventanas().horizonte,
    metodo=catalogo["regimenes"]["agregacion_horizonte"],
)

print("Reparto de la etiqueta a 21 días:")
regimenes.distribucion(regimen_futuro, etiquetador.n_estados)

## Validación visual

Esta figura decide si el etiquetado sirve. Las bandas sombreadas deben caer sobre
2008, 2011, 2020 y 2022. Si no lo hacen, el HMM ha convergido a algo no
interpretable y no tiene sentido continuar con el resto del taller.

In [ ]:
precios = datos.cargar_precios()
serie_indice = precios["sp500"].reindex(regimen_diario.index)

fig, eje = plt.subplots(figsize=(11, 5))
viz.serie_regimenes(serie_indice, regimen_diario, "Regímenes detectados sobre el S&P 500", eje=eje)
eje.axvline(pd.Timestamp(particiones.train_hasta), color=viz.TINTA_SECUNDARIA, linestyle=":", linewidth=1.2)
eje.annotate("fin de train", xy=(pd.Timestamp(particiones.train_hasta), serie_indice.min()),
             xytext=(6, 6), textcoords="offset points", fontsize=8, color=viz.TINTA_SECUNDARIA)
viz.guardar(fig, "regimenes_indice")

## Persistencia

Se guarda el etiquetador completo, no solo las etiquetas: el notebook 14 lo
necesita para etiquetar cualquier serie nueva con exactamente el mismo criterio.

In [ ]:
tabla = pd.concat([regimen_diario, regimen_futuro, proba], axis=1)
tabla.to_parquet(src.DIR_PROCESADO / "regimenes.parquet")

etiquetador.guardar(src.DIR_PROCESADO / "etiquetador_regimenes.pkl")

print("Guardadas", len(tabla), "filas ·", list(tabla.columns))

## Salidas generadas

In [ ]:
from pathlib import Path

salidas = [
    src.DIR_PROCESADO / "regimenes.parquet",
    src.DIR_PROCESADO / "etiquetador_regimenes.pkl",
    src.DIR_FIGURAS / "regimenes_indice.png",
]

for ruta in salidas:
    ruta = Path(ruta)
    marca = "ok" if ruta.exists() else "--"
    print("[{}] {}".format(marca, ruta.relative_to(src.RAIZ)))
